# Objective Quest DATAQUEST 5.0 — V15
## Fold-Safe Probabilistic Survey Denoising + Missingness-Aware Recovery Ensemble

**Frozen research contract**

V15 changes one thing relative to the established lineage: it tries to recover information removed by the survey-like ~8% cell masking pattern for four preregistered high-value questions. The reconstructor is target-independent and fold-safe. The downstream learner stays in the proven V2 ordinal GBDT family, the outer folds are the frozen V2 primary folds, and the Recovery ensemble remains the submission control.

The notebook never uses external respondent data, NHIS rows, pseudo-labels, or external labels. Historical V1–V12 artifacts are used only to reconstruct the already-LB-confirmed Recovery control and frozen folds.

A new submission is written **only** if the preregistered V15 gate passes. Otherwise the notebook exports diagnostics and the exact Recovery control only.


In [1]:
import os, gc, sys, json, math, time, random, hashlib, zipfile, warnings, traceback, platform
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import defaultdict

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

from scipy.optimize import minimize
from scipy.stats import spearmanr
from sklearn.metrics import (
    cohen_kappa_score, roc_auc_score, mean_squared_error, mean_absolute_error,
    accuracy_score, log_loss
)
from sklearn.model_selection import StratifiedShuffleSplit

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, CatBoostRegressor

try:
    import torch
    CUDA_AVAILABLE=bool(torch.cuda.is_available())
    GPU_COUNT=int(torch.cuda.device_count()) if CUDA_AVAILABLE else 0
except Exception:
    torch=None; CUDA_AVAILABLE=False; GPU_COUNT=0

SEED=42
random.seed(SEED); np.random.seed(SEED)
os.environ['PYTHONHASHSEED']=str(SEED)
os.environ.setdefault('TOKENIZERS_PARALLELISM','false')

print('python',platform.python_version())
print('numpy',np.__version__,'pandas',pd.__version__)
print('lightgbm',lgb.__version__,'xgboost',xgb.__version__)
try:
    import catboost
    print('catboost',catboost.__version__)
except Exception: pass
print('CUDA available',CUDA_AVAILABLE,'visible GPUs',GPU_COUNT)
if CUDA_AVAILABLE:
    for i in range(GPU_COUNT):
        p=torch.cuda.get_device_properties(i)
        print(i,p.name,round(p.total_memory/2**30,2),'GiB')


python 3.12.13
numpy 2.0.2 pandas 2.3.3
lightgbm 4.6.0 xgboost 3.2.0
catboost 1.2.10
CUDA available True visible GPUs 1
0 Tesla P100-PCIE-16GB 15.89 GiB


In [2]:
@dataclass(frozen=True)
class Cfg:
    version:str='v15'
    implementation_revision:str='v15.0.0'
    target:str='life_satisfaction'
    id_col:str='id'
    household_col:str='household_id'
    expected_train_rows:int=95609
    expected_train_ids:int=94663
    expected_test_rows:int=23902
    expected_test_ids:int=23666
    n_folds:int=5
    recovery_public_lb:float=0.48334
    recovery_oof_qwk:float=0.48186721909562624

    # Frozen V2-style downstream family. One primary 5-fold pass only.
    inner_holdout_frac:float=0.06
    inner_fs_top_n:int=135
    lgb_estimators:int=5500
    xgb_estimators:int=5500
    cat_iterations:int=5500
    learning_rate:float=0.025
    early_stopping:int=220

    # Target-independent reconstructor. Fixed budget: no validation-label tuning.
    recon_iterations:int=900
    recon_learning_rate:float=0.05
    recon_depth:int=7
    recon_l2:float=6.0
    recon_random_strength:float=0.5
    recon_mask_eval_rate:float=0.08

    # V2 frozen objective-diversity recipe before the tiny historical xgb-cat sidecar.
    core_weight:float=0.55
    ordinal_weight:float=0.45

    # Prespecified Recovery blend family.
    alpha_grid:tuple=(0.10,0.20,0.30,0.40)
    bootstrap_reps:int=2000

    # Submission gate frozen from the V15 research decision.
    min_delta_qwk:float=0.0010
    preferred_delta_qwk:float=0.0015
    min_positive_folds:int=4
    min_boot_p:float=0.90
    min_delta_auc_1v2:float=0.0
    min_critical_missing_delta_auc_1v2:float=0.003
    min_complete_delta_auc_1v2:float=-0.001

CFG=Cfg()
OUT=Path('/kaggle/working/oqdq5_v15') if Path('/kaggle/working').exists() else Path('./oqdq5_v15')
OUT.mkdir(parents=True,exist_ok=True)

CFG_HASH=hashlib.sha256(json.dumps(asdict(CFG),sort_keys=True,default=list).encode()).hexdigest()[:16]
print('OUT',OUT,'contract',CFG_HASH)

if CUDA_AVAILABLE and GPU_COUNT>=2:
    CAT_GPU_DEVICES='0:1'
elif CUDA_AVAILABLE:
    CAT_GPU_DEVICES='0'
else:
    CAT_GPU_DEVICES=None
print('CatBoost device contract',CAT_GPU_DEVICES)


OUT /kaggle/working/oqdq5_v15 contract 1f4d518ed9124e7a
CatBoost device contract 0


## 1. Canonical data, frozen V2 folds, and Recovery control

The unique respondent is the evaluator entity. Exact duplicate physical rows are checked for consistency and collapsed. The notebook reconstructs the known Recovery OOF/test scores from historical artifacts and asserts the canonical OOF QWK before any V15 training begins.


In [3]:
def roots():
    out=[]
    for p in [Path('/kaggle/input'),Path('/mnt/data'),Path.cwd().resolve()]:
        if p==Path('/'):
            continue
        if p.exists() and p not in out and not str(p).startswith('/proc'): out.append(p)
    return out

def locate_data():
    hits=[]
    for root in roots():
        try:
            for p in root.rglob('train_final.csv'):
                d=p.parent
                if (d/'test_final.csv').exists() and (d/'data_dictionary.csv').exists(): hits.append(d.resolve())
        except PermissionError: pass
    hits=list(dict.fromkeys(hits))
    if not hits: raise FileNotFoundError('Competition train_final.csv/test_final.csv/data_dictionary.csv not found.')
    hits.sort(key=lambda p:(not (p/'sample_submission.csv').exists(),len(str(p))))
    return hits[0]

class Store:
    def __init__(self,root=None,z=None):
        self.root=Path(root) if root else None
        self.z=Path(z) if z else None
        self.zf=zipfile.ZipFile(self.z) if self.z else None
    def csv(self,rel,**kw):
        if self.root: return pd.read_csv(self.root/rel,**kw)
        with self.zf.open(rel) as f: return pd.read_csv(f,**kw)
    def source(self): return str(self.root or self.z)
    def close(self):
        if self.zf: self.zf.close()

def locate_store():
    marker='results/v2/oof_v2.csv'; dirs=[]; zs=[]
    for root in roots():
        try:
            for p in root.rglob('oof_v2.csv'):
                if p.as_posix().endswith('/results/v2/oof_v2.csv'): dirs.append(p.parents[2].resolve())
            for z in root.rglob('*.zip'):
                try:
                    with zipfile.ZipFile(z) as q:
                        if marker in q.namelist(): zs.append(z.resolve())
                except (zipfile.BadZipFile,FileNotFoundError,PermissionError,OSError): pass
        except PermissionError: pass
    dirs=list(dict.fromkeys(dirs)); zs=list(dict.fromkeys(zs))
    if dirs: return Store(root=sorted(dirs,key=lambda p:len(str(p)))[0])
    named=[z for z in zs if z.name in {'results-v1_v12.zip','results.zip'}]
    if len(named)==1: return Store(z=named[0])
    if len(zs)==1: return Store(z=zs[0])
    if not zs: raise FileNotFoundError('Attach the historical results-v1_v12.zip Kaggle dataset.')
    raise RuntimeError(f'Ambiguous historical archives: {zs}')

DATA=locate_data(); STORE=locate_store()
train_raw=pd.read_csv(DATA/'train_final.csv',low_memory=False)
test_raw=pd.read_csv(DATA/'test_final.csv',low_memory=False)
dd=pd.read_csv(DATA/'data_dictionary.csv')

assert len(train_raw)==CFG.expected_train_rows and train_raw[CFG.id_col].nunique()==CFG.expected_train_ids
assert len(test_raw)==CFG.expected_test_rows and test_raw[CFG.id_col].nunique()==CFG.expected_test_ids
assert CFG.target in train_raw and CFG.target not in test_raw

for df,name in [(train_raw,'train'),(test_raw,'test')]:
    dup=df[df.duplicated(CFG.id_col,False)]
    if len(dup):
        g=dup.groupby(CFG.id_col,sort=False)
        bad=[c for c in df.columns if c!=CFG.id_col and g[c].nunique(dropna=False).max()>1]
        if bad: raise RuntimeError(f'{name} duplicate IDs disagree: {bad[:10]}')

v2meta=STORE.csv('results/v2/oof_v2.csv',usecols=['id','life_satisfaction','primary_fold'])
canon=v2meta.drop_duplicates('id',keep='first').reset_index(drop=True)
assert len(canon)==CFG.expected_train_ids and set(canon.primary_fold.astype(int))==set(range(5))
train=canon[['id']].merge(train_raw.drop_duplicates('id',keep='first'),on='id',validate='one_to_one')
train['__fold__']=canon.primary_fold.astype(np.int8).to_numpy()
test=test_raw.drop_duplicates('id',keep='first').reset_index(drop=True)
assert np.array_equal(train[CFG.target].astype(int),canon[CFG.target].astype(int))
assert set(train.id).isdisjoint(set(test.id))

COMP={
'v2_final':('v2','oof_v2.csv','test_predictions_v2.csv','oof_final_continuous','test_final_continuous'),
'v5_final':('v5_1','oof_v5.csv','test_predictions_v5.csv','oof_final_honest_continuous','pred_final_continuous'),
'v61_final':('v6_1','oof_v6.csv','test_predictions_v6.csv','oof_final_crossfit','pred_final_continuous'),
'v7_stack':('v7','oof_v7.csv','test_scores_v7.csv','v7_stack','v7_stack')}
META={'v2':('primary_fold','life_satisfaction'),'v5_1':('fold','life_satisfaction'),'v6_1':('primary_fold','life_satisfaction'),'v7':('primary_fold','life_satisfaction')}
train_ids=train.id.to_numpy(); test_ids=test.id.to_numpy(); y=train[CFG.target].astype(int).to_numpy(); folds=train.__fold__.astype(int).to_numpy()

def test_ids_for(df):
    if 'id' in df: return df.id
    if 'test_feature_id' in df: return df.test_feature_id
    if 'row_index' in df:
        if not np.array_equal(df.row_index.to_numpy(),np.arange(len(test_raw))): raise RuntimeError('Historical row_index drift.')
        return test_raw.id.reset_index(drop=True)
    raise KeyError('No historical test ID mapping.')

def load_component(spec):
    v,of,tf,oc,tc=spec; od=STORE.csv(f'results/{v}/{of}'); td=STORE.csv(f'results/{v}/{tf}')
    fc,yc=META[v]; g=od[['id',yc,fc,oc]].groupby('id',sort=False)
    assert g[yc].nunique().max()==1 and g[fc].nunique().max()==1 and (g[oc].max()-g[oc].min()).max()<1e-8
    a=g.agg({yc:'first',oc:'mean'}).reset_index(); a=canon[['id']].merge(a,on='id',validate='one_to_one')
    assert np.array_equal(a[yc].astype(int),y)
    td=td.copy(); td['_id']=test_ids_for(td).to_numpy(); h=td[['_id',tc]].groupby('_id',sort=False)
    assert (h[tc].max()-h[tc].min()).max()<1e-8
    b=h[tc].mean().reset_index().rename(columns={'_id':'id'}); b=test[['id']].merge(b,on='id',validate='one_to_one')
    return a[oc].to_numpy(float),b[tc].to_numpy(float)

arr={n:load_component(s) for n,s in COMP.items()}
REC_NAMES=('v2_final','v5_final','v61_final','v7_stack')
RW={0:(0.6131481145225551,0.1984633896432871,0.18838849583415773,0.0),1:(0.5736087705637004,0.2130793852900037,0.20117728057551384,0.012134563570782105),2:(0.5260926035320435,0.22651646669428574,0.20045779567769734,0.04693313409597358),3:(0.412997730762594,0.12147998345945334,0.2995554634343377,0.16596682234361498),4:(0.4419367210113302,0.27031416820745807,0.2416404622741877,0.04610864850702414)}
RT={0:(1.5694444444444446,2.011875,2.4941666666666666),1:(1.57,2.011875,2.5100000000000002),2:(1.57,2.0162500000000003,2.5100000000000002),3:(1.5683333333333334,2.011875,2.524166666666667),4:(1.568888888888889,2.014375,2.5100000000000002)}
RFW=np.array((0.5333147962860486,0.2160045364556684,0.20393903979487613,0.0467416274634069))
RFT=np.array((1.5694444444444446,2.011875,2.5100000000000002))
RX=np.column_stack([arr[n][0] for n in REC_NAMES]); RTX=np.column_stack([arr[n][1] for n in REC_NAMES])
recovery_oof=np.empty(len(y)); recovery_class=np.empty(len(y),int)
for f in range(5):
    m=folds==f; recovery_oof[m]=RX[m]@np.array(RW[f]); t=np.array(RT[f])
    recovery_class[m]=1+(recovery_oof[m]>=t[0])+(recovery_oof[m]>=t[1])+(recovery_oof[m]>=t[2])
recovery_test=RTX@RFW
recovery_test_class=1+(recovery_test>=RFT[0])+(recovery_test>=RFT[1])+(recovery_test>=RFT[2])
RECOVERY_QWK=float(cohen_kappa_score(y,recovery_class,weights='quadratic'))
assert abs(RECOVERY_QWK-CFG.recovery_oof_qwk)<1e-12

control=pd.DataFrame({'id':test_ids,'life_satisfaction':recovery_test_class.astype(int)})
assert len(control)==CFG.expected_test_ids and control.id.is_unique
control.to_csv(OUT/'submission_v15_recovery_control.csv',index=False)
print('DATA',DATA,'HISTORY',STORE.source())
print('unique respondents',len(train),len(test),'Recovery OOF',RECOVERY_QWK,'public LB',CFG.recovery_public_lb)


DATA /kaggle/input/competitions/oq-dq-5 HISTORY /kaggle/input/datasets/riantamaputra/dataquest-2026-historical-results
unique respondents 94663 23666 Recovery OOF 0.48186721909562624 public LB 0.48334


## 2. Fold-local deterministic repair and survey semantics

V15 first applies the same conservative deterministic repair family as V2. Rules are learned only from the outer-training fold. Raw values are retained; repair fills only missing destination cells. The later probabilistic reconstructor operates only on critical cells that are still missing after these repairs.


In [4]:
REPAIR_PAIRS=[
('data_year','survey_year'),('survey_year','data_year'),('interview_month','interview_quarter'),
('food_security_4cat','food_security_3cat'),('cancer_types_count','cancer_ever'),('ecigarette_status','ever_used_ecigarette'),
('children_in_family','children_in_household'),('children_in_household','children_in_family'),('smoking_status','ever_smoked'),
('health_insurance_type','private_health_insurance'),('marital_status_detailed','marital_status')]
SPECIAL_CODES={'weight_pounds':{996,997,999},'height_inches':{96,97,98,99},'age':{97,99},'hours_worked_per_week':{990},'days_missed_work':{997,998,999},'income_to_poverty_ratio':{110}}

NATIVE_NOMINAL_COLS=[
'record_type','survey_year','geographic_region','pseudo_stratum','pseudo_psu','sex','citizenship_status','born_in_country',
'sexual_orientation','parental_status','relationship_to_child','marital_status','legal_marital_status','marital_status_detailed',
'health_insurance_type','insurance_source_plan1','type_of_care_provider','home_ownership','smoking_status','ecigarette_status',
'interview_quarter','interview_month']

FUNCTIONAL_DIFFICULTY_COLS=['difficulty_walking_steps','difficulty_walking_short','difficulty_walking_medium','difficulty_climbing_steps','difficulty_using_hands','difficulty_raising_arm','difficulty_self_care','difficulty_communicating','difficulty_remembering','difficulty_errands_alone','difficulty_social_activities']
CHRONIC_CONDITION_COLS=['hypertension_ever','high_cholesterol_ever','heart_disease_ever','heart_attack_ever','angina_ever','stroke_ever','asthma_ever','diabetes_ever','prediabetes_ever','cancer_ever','lung_disease_ever','arthritis_ever','dementia_ever','chronic_fatigue_ever']
COST_BARRIER_YES_COLS=['trouble_paying_medical_bills','delayed_care_due_to_cost','skipped_care_due_to_cost','skipped_medication_to_save_money','took_less_medication_to_save_money','delayed_prescription_to_save_money','skipped_prescription_due_to_cost','delayed_therapy_due_to_cost','skipped_therapy_due_to_cost']
INCOME_SOURCE_COLS=['has_wage_income','has_investment_income','has_social_security_income','has_disability_income','has_public_assistance_income','has_retirement_income','has_other_income']
INSURANCE_YES_COLS=['has_health_insurance','has_private_insurance','has_senior_gov_insurance','has_lowincome_gov_insurance','has_child_gov_insurance','has_military_insurance','has_state_health_plan','has_other_gov_insurance']
FREQUENCY_COLS=['pseudo_stratum','pseudo_psu']

DOMAIN_FORCE_KEEP=[
'general_health_status','general_health_status__semantic','depression_frequency','depression_frequency__semantic','depression_burden',
'anxiety_frequency','anxiety_frequency__semantic','anxiety_burden','anxiety_level','income_to_poverty_ratio','income_poverty_category','income_poverty_category__semantic',
'food_security_4cat','food_security_4cat__semantic','food_security_3cat','difficulty_social_activities','difficulty_remembering','marital_status','marital_status_detailed',
'age','survey_weight','row_missing_count_raw','row_missing_count_after_repair','repaired_value_count','special_code_count','nonstandard_code_count',
'functional_difficulty_count','chronic_condition_count','cost_barrier_count','food_insecurity_count','income_source_count','insurance_count','household_size_proxy',
'mental_frequency_burden','health_x_depression_burden','health_x_anxiety_burden','poverty_x_food_security','mental_x_functional_burden']


def learn_mode_mapping(df,src,dst,min_support=100):
    if src not in df or dst not in df: return None
    p=df[[src,dst]].dropna()
    if len(p)<min_support: return None
    mp=p.groupby(src)[dst].agg(lambda s:s.mode().iloc[0] if not s.mode().empty else np.nan).to_dict()
    pred=p[src].map(mp)
    return {'mapping':mp,'accuracy':float((pred.values==p[dst].values).mean()),'support':len(p)}

class FoldPreprocessor:
    def __init__(self,min_accuracy=.995,min_support=100):
        self.min_accuracy=min_accuracy; self.min_support=min_support; self.rules={}; self.report=None
    def fit(self,df):
        rows=[]
        for src,dst in REPAIR_PAIRS:
            q=learn_mode_mapping(df,src,dst,self.min_support)
            if q is None: continue
            ok=q['accuracy']>=self.min_accuracy
            rows.append({'source':src,'destination':dst,'accuracy':q['accuracy'],'support':q['support'],'accepted':ok})
            if ok: self.rules[(src,dst)]=q['mapping']
        self.report=pd.DataFrame(rows); return self
    def transform(self,df):
        out=df.drop(columns=[CFG.target,'__fold__'],errors='ignore').copy()
        base=[c for c in out.columns if c not in [CFG.id_col,CFG.household_col] and not str(c).startswith('__')]
        out['row_missing_count_raw']=out[base].isna().sum(axis=1).astype(np.int16)
        before=out[base].isna().sum(axis=1)
        for (src,dst),mp in self.rules.items():
            if src not in out or dst not in out: continue
            m=out[dst].isna() & out[src].notna()
            if m.any():
                v=out.loc[m,src].map(mp); idx=v[v.notna()].index; out.loc[idx,dst]=v.loc[idx]
        after=out[base].isna().sum(axis=1)
        out['row_missing_count_after_repair']=after.astype(np.int16)
        out['repaired_value_count']=(before-after).astype(np.int16)
        flags=[]
        for c,codes in SPECIAL_CODES.items():
            if c in out:
                n=f'{c}__special_code'; out[n]=out[c].isin(codes).astype(np.int8); flags.append(n)
        out['special_code_count']=out[flags].sum(axis=1).astype(np.int16) if flags else 0
        return out


def parse_valid_codes(x):
    import re
    if pd.isna(x): return None
    s=str(x).strip()
    if not s or s.lower() in {'numerik','nilai tahun','kode numerik'}: return None
    m=re.findall(r'(?:^|,)\s*(-?\d+(?:\.\d+)?)\s*=',s)
    if m:
        vals=sorted({float(z) for z in m}); return vals if 1<=len(vals)<=30 else None
    m2=re.fullmatch(r'\s*(-?\d+)\s*-\s*(-?\d+)\+?\s*',s)
    if m2:
        lo,hi=map(int,m2.groups()); return [float(z) for z in range(lo,hi+1)] if 0<=hi-lo<=30 else None
    parts=[p.strip() for p in s.split(',')]
    if 1<len(parts)<=30:
        try: return sorted({float(p) for p in parts})
        except Exception: return None
    return None

DICT_VALID={}
for _,r in dd.iterrows():
    c=str(r['kolom']); vals=parse_valid_codes(r['nilai'])
    if c in test.columns and vals is not None: DICT_VALID[c]=set(vals)

# Freeze V2 semantic-copy list from historical V2 audit rather than inspecting current validation/test values.
try:
    SEMANTIC_CODE_COLS=STORE.csv('results/v2/semantic_code_audit_v2.csv')['feature'].astype(str).tolist()
except Exception:
    SEMANTIC_CODE_COLS=[]
print('V2 semantic-copy features',len(SEMANTIC_CODE_COLS))


V2 semantic-copy features 150


In [5]:
def count_yes(df,cols):
    cols=[c for c in cols if c in df]
    return (df[cols]==1).sum(axis=1).astype(np.int16) if cols else pd.Series(0,index=df.index,dtype=np.int16)

def reverse_ordinal(s,max_valid):
    x=pd.to_numeric(s,errors='coerce'); out=pd.Series(np.nan,index=s.index,dtype=float)
    m=x.between(1,max_valid); out.loc[m]=(max_valid+1)-x.loc[m]; return out

class SemanticFeatureEngineer:
    def __init__(self,valid): self.valid=valid; self.freq={}; self.engineered=[]
    def fit(self,df):
        for c in FREQUENCY_COLS:
            if c in df: self.freq[c]=df[c].value_counts(dropna=False).to_dict()
        return self
    def transform(self,df):
        out=df.drop(columns=[c for c in df.columns if str(c).startswith('__')],errors='ignore').copy(); before=set(out.columns)
        for c,mp in self.freq.items():
            if c in out:
                out[f'{c}__frequency']=out[c].map(mp).fillna(0).astype(np.float32)
                out[f'{c}__log_frequency']=np.log1p(out[f'{c}__frequency']).astype(np.float32)
        nonstd=[]
        for c in SEMANTIC_CODE_COLS:
            if c not in out or c not in self.valid: continue
            raw=pd.to_numeric(out[c],errors='coerce'); valid=self.valid[c]
            out[f'{c}__semantic']=raw.where(raw.isin(valid)).astype(np.float32)
            n=f'{c}__nonstandard'; out[n]=(raw.notna() & ~raw.isin(valid)).astype(np.int8); nonstd.append(n)
        out['nonstandard_code_count']=out[nonstd].sum(axis=1).astype(np.int16) if nonstd else 0
        out['functional_difficulty_count']=count_yes(out,FUNCTIONAL_DIFFICULTY_COLS)
        out['chronic_condition_count']=count_yes(out,CHRONIC_CONDITION_COLS)
        out['cost_barrier_count']=count_yes(out,COST_BARRIER_YES_COLS)
        out['income_source_count']=count_yes(out,INCOME_SOURCE_COLS)
        out['insurance_count']=count_yes(out,INSURANCE_YES_COLS)
        food=[reverse_ordinal(out[c],3).fillna(0) for c in ['worried_food_would_run_out','food_did_not_last','could_not_afford_balanced_meals'] if c in out]
        out['food_insecurity_count']=sum(food).astype(np.float32) if food else 0.0
        if 'received_food_assistance' in out: out['food_insecurity_count']+=(out['received_food_assistance']==1).astype(np.float32)
        if 'depression_frequency' in out: out['depression_burden']=reverse_ordinal(out['depression_frequency'],5).astype(np.float32)
        if 'anxiety_frequency' in out: out['anxiety_burden']=reverse_ordinal(out['anxiety_frequency'],5).astype(np.float32)
        mental=[out[c].fillna(0) for c in ['depression_burden','anxiety_burden'] if c in out]
        out['mental_frequency_burden']=sum(mental).astype(np.float32) if mental else 0.0
        hh=[pd.to_numeric(out[c],errors='coerce').fillna(0) for c in ['children_in_household','adults_in_household'] if c in out]
        out['household_size_proxy']=sum(hh).astype(np.float32) if hh else 0.0
        if 'age' in out:
            a=pd.to_numeric(out['age'],errors='coerce'); a=a.where(a.between(18,85))
            out['age_band']=pd.cut(a,bins=[17,24,34,44,54,64,74,85],labels=False,include_lowest=True).astype(float)
        def prod(a,b,n):
            if a in out and b in out: out[n]=(pd.to_numeric(out[a],errors='coerce')*pd.to_numeric(out[b],errors='coerce')).astype(np.float32)
        prod('general_health_status__semantic','depression_burden','health_x_depression_burden')
        prod('general_health_status__semantic','anxiety_burden','health_x_anxiety_burden')
        prod('income_poverty_category__semantic','food_security_4cat__semantic','poverty_x_food_security')
        prod('mental_frequency_burden','functional_difficulty_count','mental_x_functional_burden')
        out=out.drop(columns=[c for c in [CFG.id_col,CFG.household_col] if c in out],errors='ignore')
        self.engineered=[c for c in out.columns if c not in before]
        return out


## 3. Preregistered critical-question posterior embedding

The four reconstruction targets are fixed before the run. Survey special codes are retained as categorical outcomes, but posterior expected values are computed only over semantically valid ordered response codes. This avoids treating refusal/don't-know codes such as 7/8/9 as ordinal severity.


In [6]:
TARGET_SPECS={
'general_health_status':{'classes':[1.,2.,3.,4.,5.,7.,9.],'valid':[1.,2.,3.,4.,5.]},
'depression_frequency':{'classes':[1.,2.,3.,4.,5.,7.,8.,9.],'valid':[1.,2.,3.,4.,5.]},
'worry_about_medical_costs':{'classes':[1.,2.,3.,7.,8.,9.],'valid':[1.,2.,3.]},
'food_security_4cat':{'classes':[1.,2.,3.,4.,8.],'valid':[1.,2.,3.,4.]}}
CRITICAL=list(TARGET_SPECS)
POSTERIOR_PREFIX='v15post__'


def cat_frame(df,features):
    X=df[features].copy(); cats=[c for c in NATIVE_NOMINAL_COLS if c in X]
    for c in cats: X[c]=X[c].map(lambda v:'__MISSING__' if pd.isna(v) else str(v))
    return X,cats

def fit_reconstructor(base_tr,target_col,seed):
    obs=base_tr[target_col].notna()
    features=[c for c in base_tr.columns if c not in [target_col,CFG.id_col,CFG.household_col] and not str(c).startswith('__')]
    X,cats=cat_frame(base_tr.loc[obs],features); yy=base_tr.loc[obs,target_col].astype(str)
    params=dict(
        loss_function='MultiClass',iterations=CFG.recon_iterations,learning_rate=CFG.recon_learning_rate,
        depth=CFG.recon_depth,l2_leaf_reg=CFG.recon_l2,random_strength=CFG.recon_random_strength,
        random_seed=seed,allow_writing_files=False,verbose=False,thread_count=-1
    )
    attempts=[]
    if CUDA_AVAILABLE:
        if CAT_GPU_DEVICES: attempts.append(('GPU',CAT_GPU_DEVICES))
        if CAT_GPU_DEVICES!='0': attempts.append(('GPU','0'))
    attempts.append(('CPU',None))
    last=None
    for task,dev in attempts:
        try:
            q=params.copy(); q['task_type']=task
            if dev is not None: q['devices']=dev
            m=CatBoostClassifier(**q); m.fit(X,yy,cat_features=cats)
            return m,features,cats,task,dev
        except Exception as e:
            last=e; print('reconstructor fallback',target_col,task,dev,repr(e)); gc.collect()
            if torch is not None and CUDA_AVAILABLE: torch.cuda.empty_cache()
    raise last

def predict_proba_aligned(model,base,features,cats,spec,rows=None):
    d=base if rows is None else base.loc[rows]
    X,_=cat_frame(d,features)
    p=np.asarray(model.predict_proba(X),float)
    model_classes=[float(x) for x in model.classes_]
    full=np.zeros((len(d),len(spec['classes'])),dtype=np.float32)
    pos={float(v):i for i,v in enumerate(spec['classes'])}
    for j,c in enumerate(model_classes):
        if c in pos: full[:,pos[c]]=p[:,j]
    s=full.sum(axis=1,keepdims=True); full=np.divide(full,s,out=np.zeros_like(full),where=s>0)
    return full

def add_posterior_features(base,raw_before_repair,target_col,spec,model,features,cats):
    out=base.copy(); vals=pd.to_numeric(out[target_col],errors='coerce')
    raw_missing=raw_before_repair[target_col].isna().to_numpy()
    missing=vals.isna().to_numpy()
    classes=np.asarray(spec['classes'],float); valid=np.asarray(spec['valid'],float)
    probs=np.zeros((len(out),len(classes)),dtype=np.float32)
    observed=~missing
    pos={float(v):i for i,v in enumerate(classes)}
    if observed.any():
        for code,j in pos.items(): probs[observed & (vals.to_numpy()==code),j]=1.0
        unmatched=observed & (probs.sum(axis=1)==0)
        if unmatched.any():
            # Preserve unknown observed codes as uncertainty rather than inventing an ordered value.
            probs[unmatched]=1.0/len(classes)
    if missing.any(): probs[missing]=predict_proba_aligned(model,out,features,cats,spec,rows=out.index[missing])
    eps=1e-12
    for j,code in enumerate(classes): out[f'{POSTERIOR_PREFIX}{target_col}__p_{int(code)}']=probs[:,j]
    ent=-(probs*np.log(np.clip(probs,eps,1))).sum(axis=1)
    conf=probs.max(axis=1)
    valid_mask=np.isin(classes,valid); valid_mass=probs[:,valid_mask].sum(axis=1)
    expected=(probs[:,valid_mask]*classes[valid_mask]).sum(axis=1)
    expected=np.divide(expected,valid_mass,out=np.full(len(out),np.nan,dtype=float),where=valid_mass>eps)
    out[f'{POSTERIOR_PREFIX}{target_col}__expected_valid']=expected.astype(np.float32)
    out[f'{POSTERIOR_PREFIX}{target_col}__valid_mass']=valid_mass.astype(np.float32)
    out[f'{POSTERIOR_PREFIX}{target_col}__nonstandard_mass']=(1-valid_mass).astype(np.float32)
    out[f'{POSTERIOR_PREFIX}{target_col}__entropy']=ent.astype(np.float32)
    out[f'{POSTERIOR_PREFIX}{target_col}__confidence']=conf.astype(np.float32)
    out[f'{POSTERIOR_PREFIX}{target_col}__raw_missing']=raw_missing.astype(np.int8)
    out[f'{POSTERIOR_PREFIX}{target_col}__reconstructed']=missing.astype(np.int8)
    return out,missing


## 4. Leakage-safe downstream V2-style ordinal GBDT

Feature selection and early stopping are performed on a fixed inner holdout drawn only from the outer-training fold. The outer validation labels are never used for feature selection, model stopping, blend-alpha choice, or threshold fitting for that fold.


In [7]:
def qwk(a,b): return float(cohen_kappa_score(np.asarray(a),np.asarray(b),weights='quadratic'))
def disc(p,t):
    t=np.sort(np.asarray(t,float)); p=np.asarray(p,float)
    return (1+(p>=t[0])+(p>=t[1])+(p>=t[2])).astype(int)

def initial_thr(yy,p):
    vc=pd.Series(yy).value_counts(normalize=True).sort_index(); cum=np.cumsum([vc.get(k,0.) for k in [1,2,3]])
    return np.quantile(p,np.clip(cum,.001,.999))
def fit_thr(yy,p,maxiter=160):
    starts=[initial_thr(yy,p),np.array([1.55,2.03,2.52])]; best=None
    for st in starts:
        def obj(t):
            z=np.sort(t)
            if np.min(np.diff(z))<.03: return 1e3
            return -qwk(yy,disc(p,z))
        r=minimize(obj,np.sort(st),method='Nelder-Mead',options={'maxiter':maxiter,'xatol':1e-5,'fatol':1e-7})
        z=np.sort(r.x); s=qwk(yy,disc(p,z))
        if best is None or s>best[1]: best=(z,s)
    return best[0]

def auc12(yy,p):
    m=np.isin(yy,[1,2]); return float(roc_auc_score((yy[m]>1).astype(int),np.asarray(p)[m]))
def aucgt1(yy,p): return float(roc_auc_score((np.asarray(yy)>1).astype(int),np.asarray(p)))

def inner_split(ytr,seed):
    s=StratifiedShuffleSplit(n_splits=1,test_size=CFG.inner_holdout_frac,random_state=seed)
    a,b=next(s.split(np.zeros(len(ytr)),ytr)); return a,b

def make_fs(seed,use_gpu):
    q=dict(objective='regression_l2',n_estimators=1800,learning_rate=.035,num_leaves=31,min_child_samples=40,colsample_bytree=.9,subsample=.9,subsample_freq=1,reg_lambda=4.,random_state=seed,n_jobs=-1,verbosity=-1)
    if use_gpu: q.update(device_type='gpu',max_bin=63)
    return lgb.LGBMRegressor(**q)

def select_features(X,ytr,itr,iva,seed,posterior_cols):
    allf=X.columns.tolist(); m=None
    for gpu in ([True,False] if CUDA_AVAILABLE else [False]):
        try:
            m=make_fs(seed,gpu); m.fit(X.iloc[itr],ytr[itr],eval_set=[(X.iloc[iva],ytr[iva])],callbacks=[lgb.early_stopping(120,verbose=False)]); break
        except Exception as e:
            if gpu: print('FS LightGBM GPU fallback',repr(e)); gc.collect(); continue
            raise
    gain=m.booster_.feature_importance(importance_type='gain')
    ranked=[f for _,f in sorted(zip(gain,allf),reverse=True)]
    mandatory=[c for c in DOMAIN_FORCE_KEEP if c in X]+[c for c in posterior_cols if c in X]
    # The V15 audit explicitly prevents the old V2 mandatory-feature explosion.
    selected=list(dict.fromkeys(ranked[:CFG.inner_fs_top_n]+mandatory))
    return selected,pd.DataFrame({'feature':allf,'gain':gain}).sort_values('gain',ascending=False)

def lgb_reg(seed,use_gpu,n_est=None):
    q=dict(objective='regression_l2',n_estimators=n_est or CFG.lgb_estimators,learning_rate=CFG.learning_rate,num_leaves=31,min_child_samples=40,subsample=.85,subsample_freq=1,colsample_bytree=.85,reg_alpha=.05,reg_lambda=4.,random_state=seed,n_jobs=-1,verbosity=-1)
    if use_gpu: q.update(device_type='gpu',max_bin=63)
    return lgb.LGBMRegressor(**q)
def lgb_bin(seed,use_gpu,n_est=None):
    q=dict(objective='binary',n_estimators=n_est or CFG.lgb_estimators,learning_rate=CFG.learning_rate,num_leaves=31,min_child_samples=40,subsample=.85,subsample_freq=1,colsample_bytree=.85,reg_lambda=4.,random_state=seed,n_jobs=-1,verbosity=-1)
    if use_gpu: q.update(device_type='gpu',max_bin=63)
    return lgb.LGBMClassifier(**q)
def xgb_reg(seed,use_gpu,n_est=None):
    q=dict(objective='reg:squarederror',eval_metric='rmse',n_estimators=n_est or CFG.xgb_estimators,learning_rate=CFG.learning_rate,max_depth=6,min_child_weight=8,subsample=.85,colsample_bytree=.85,reg_alpha=.05,reg_lambda=4.,tree_method='hist',random_state=seed,n_jobs=-1,early_stopping_rounds=CFG.early_stopping)
    if use_gpu: q['device']='cuda:0'
    return xgb.XGBRegressor(**q)

def fit_lgb_reg_inner(A,ytr,B,C,itr,iva,seed):
    for gpu in ([True,False] if CUDA_AVAILABLE else [False]):
        try:
            m=lgb_reg(seed,gpu); m.fit(A.iloc[itr],ytr[itr],eval_set=[(A.iloc[iva],ytr[iva])],callbacks=[lgb.early_stopping(CFG.early_stopping,verbose=False)])
            return m.predict(B,num_iteration=m.best_iteration_),m.predict(C,num_iteration=m.best_iteration_),int(m.best_iteration_)
        except Exception as e:
            if gpu: print('LGB fallback',repr(e)); gc.collect(); continue
            raise

def fit_xgb_inner(A,ytr,B,C,itr,iva,seed):
    for gpu in ([True,False] if CUDA_AVAILABLE else [False]):
        try:
            m=xgb_reg(seed,gpu); m.fit(A.iloc[itr],ytr[itr],eval_set=[(A.iloc[iva],ytr[iva])],verbose=False)
            return m.predict(B),m.predict(C),int(getattr(m,'best_iteration',CFG.xgb_estimators-1)+1)
        except Exception as e:
            if gpu: print('XGB fallback',repr(e)); gc.collect(); continue
            raise

def fit_cat_inner(A,ytr,B,C,itr,iva,seed):
    A2,cats=cat_frame(A,A.columns.tolist()); B2,_=cat_frame(B,B.columns.tolist()); C2,_=cat_frame(C,C.columns.tolist())
    q=dict(loss_function='RMSE',eval_metric='RMSE',iterations=CFG.cat_iterations,learning_rate=CFG.learning_rate,depth=7,l2_leaf_reg=6.,random_strength=.5,random_seed=seed,od_type='Iter',od_wait=CFG.early_stopping,allow_writing_files=False,verbose=False,thread_count=-1)
    attempts=[]
    if CUDA_AVAILABLE:
        if CAT_GPU_DEVICES: attempts.append(('GPU',CAT_GPU_DEVICES))
        if CAT_GPU_DEVICES!='0': attempts.append(('GPU','0'))
    attempts.append(('CPU',None)); last=None
    for task,dev in attempts:
        try:
            p=q.copy(); p['task_type']=task
            if dev is not None: p['devices']=dev
            m=CatBoostRegressor(**p); m.fit(A2.iloc[itr],ytr[itr],eval_set=(A2.iloc[iva],ytr[iva]),cat_features=cats,use_best_model=True)
            bi=int(m.get_best_iteration())+1
            return np.asarray(m.predict(B2),float),np.asarray(m.predict(C2),float),bi
        except Exception as e:
            last=e; print('CatBoost reg fallback',task,dev,repr(e)); gc.collect()
    raise last

def fit_ordinal_inner(A,ytr,B,C,itr,iva,seed):
    vb=[]; tb=[]; bis=[]
    for boundary in [1,2,3]:
        yy=(ytr>boundary).astype(int); done=False
        for gpu in ([True,False] if CUDA_AVAILABLE else [False]):
            try:
                m=lgb_bin(seed+101*boundary,gpu); m.fit(A.iloc[itr],yy[itr],eval_set=[(A.iloc[iva],yy[iva])],callbacks=[lgb.early_stopping(CFG.early_stopping,verbose=False)])
                vb.append(m.predict_proba(B,num_iteration=m.best_iteration_)[:,1]); tb.append(m.predict_proba(C,num_iteration=m.best_iteration_)[:,1]); bis.append(int(m.best_iteration_)); done=True; break
            except Exception as e:
                if gpu: print('Ordinal LGB fallback',boundary,repr(e)); gc.collect(); continue
                raise
        assert done
    vb=np.column_stack(vb); tb=np.column_stack(tb)
    vb[:,1]=np.minimum(vb[:,1],vb[:,0]); vb[:,2]=np.minimum(vb[:,2],vb[:,1])
    tb[:,1]=np.minimum(tb[:,1],tb[:,0]); tb[:,2]=np.minimum(tb[:,2],tb[:,1])
    return 1+vb.sum(1),1+tb.sum(1),bis


## 5. Five-fold V15 execution

Each outer fold independently learns deterministic repairs, four target-independent reconstructors, posterior features, fold-local semantic features, nested feature selection, and the fixed V2-style GBDT predictor. Fold artifacts are persisted immediately to make Kaggle reruns recoverable.


In [8]:
def fold_paths(f):
    return OUT/f'fold{f}_predictions.npz',OUT/f'fold{f}_meta.json',OUT/f'fold{f}_reconstruction.csv',OUT/f'fold{f}_features.csv'

def valid_cached_fold(f,va_ids):
    pp,mp,_,_=fold_paths(f)
    if not pp.exists() or not mp.exists(): return False
    try:
        meta=json.load(open(mp)); z=np.load(pp,allow_pickle=False)
        return meta.get('contract_hash')==CFG_HASH and np.array_equal(z['valid_id'],va_ids) and np.array_equal(z['test_id'],test_ids)
    except Exception: return False

for f in range(CFG.n_folds):
    print('\n'+'='*90,'\nFOLD',f,'\n'+'='*90)
    tr_idx=np.where(folds!=f)[0]; va_idx=np.where(folds==f)[0]
    va_ids=train_ids[va_idx]
    pp,mp,rp,fp=fold_paths(f)
    if valid_cached_fold(f,va_ids):
        print('Using validated cached fold artifact',pp); continue

    raw_tr=train.iloc[tr_idx].copy(); raw_va=train.iloc[va_idx].copy(); raw_te=test.copy()
    ytr=y[tr_idx]

    pre=FoldPreprocessor().fit(raw_tr)
    btr=pre.transform(raw_tr); bva=pre.transform(raw_va); bte=pre.transform(raw_te)
    rec_rows=[]; gate_va=np.zeros(len(bva),dtype=bool); gate_te=np.zeros(len(bte),dtype=bool)

    for j,target_col in enumerate(CRITICAL):
        spec=TARGET_SPECS[target_col]; seed=SEED+f*1009+j*131
        model,features,cats,task,dev=fit_reconstructor(btr,target_col,seed)

        # Preregistered artificial-mask diagnostic: deterministic 8% sample of observed outer-validation cells.
        obs=np.where(bva[target_col].notna().to_numpy())[0]
        rng=np.random.default_rng(seed+777); n=max(1,int(round(len(obs)*CFG.recon_mask_eval_rate)))
        probe=np.sort(rng.choice(obs,size=min(n,len(obs)),replace=False))
        pr=predict_proba_aligned(model,bva,features,cats,spec,rows=bva.index[probe])
        pred_code=np.asarray(spec['classes'])[pr.argmax(1)]
        true_code=bva.iloc[probe][target_col].to_numpy(float)
        majority=float(btr.loc[btr[target_col].notna(),target_col].mode().iloc[0])
        pos={float(c):k for k,c in enumerate(spec['classes'])}
        known=np.array([x in pos for x in true_code])
        ll=np.nan
        if known.any(): ll=float(log_loss([pos[x] for x in true_code[known]],pr[known],labels=list(range(len(spec['classes'])))))
        rec_rows.append({'fold':f,'feature':target_col,'observed_outer_train':int(btr[target_col].notna().sum()),'remaining_missing_train':int(btr[target_col].isna().sum()),'remaining_missing_valid':int(bva[target_col].isna().sum()),'remaining_missing_test':int(bte[target_col].isna().sum()),'mask_probe_n':len(probe),'accuracy':float(accuracy_score(true_code,pred_code)),'majority_accuracy':float(np.mean(true_code==majority)),'log_loss':ll,'backend':task,'devices':str(dev)})

        btr,miss_tr=add_posterior_features(btr,raw_tr.drop(columns=[CFG.target],errors='ignore'),target_col,spec,model,features,cats)
        bva,miss_va=add_posterior_features(bva,raw_va.drop(columns=[CFG.target],errors='ignore'),target_col,spec,model,features,cats)
        bte,miss_te=add_posterior_features(bte,raw_te,target_col,spec,model,features,cats)
        gate_va |= miss_va; gate_te |= miss_te
        del model; gc.collect()
        if torch is not None and CUDA_AVAILABLE: torch.cuda.empty_cache()

    fe=SemanticFeatureEngineer(DICT_VALID).fit(btr)
    Xtr=fe.transform(btr); Xva=fe.transform(bva); Xte=fe.transform(bte)
    assert Xtr.columns.tolist()==Xva.columns.tolist()==Xte.columns.tolist()
    posterior_cols=[c for c in Xtr.columns if c.startswith(POSTERIOR_PREFIX)]
    itr,iva=inner_split(ytr,SEED+f*4099)
    selected,fsimp=select_features(Xtr,ytr,itr,iva,SEED+f*5099,posterior_cols)
    A=Xtr[selected].copy(); B=Xva[selected].copy(); C=Xte[selected].copy()

    pl,p_lt,bi_l=fit_lgb_reg_inner(A,ytr,B,C,itr,iva,SEED+f*7001+1)
    px,p_xt,bi_x=fit_xgb_inner(A,ytr,B,C,itr,iva,SEED+f*7001+2)
    pc,p_ct,bi_c=fit_cat_inner(A,ytr,B,C,itr,iva,SEED+f*7001+3)
    po,p_ot,bi_o=fit_ordinal_inner(A,ytr,B,C,itr,iva,SEED+f*7001+4)
    core_v=(pl+px+pc)/3.; core_t=(p_lt+p_xt+p_ct)/3.
    den_v=CFG.core_weight*core_v+CFG.ordinal_weight*po
    den_t=CFG.core_weight*core_t+CFG.ordinal_weight*p_ot
    assert np.isfinite(den_v).all() and np.isfinite(den_t).all()

    np.savez_compressed(pp,valid_id=va_ids,valid_pred=den_v,test_id=test_ids,test_pred=den_t,valid_gate=gate_va.astype(np.int8),test_gate=gate_te.astype(np.int8))
    pd.DataFrame(rec_rows).to_csv(rp,index=False)
    fsimp.assign(selected=fsimp.feature.isin(selected)).to_csv(fp,index=False)
    meta={'contract_hash':CFG_HASH,'fold':f,'train_n':len(tr_idx),'valid_n':len(va_idx),'selected_n':len(selected),'posterior_n':len(posterior_cols),'lgb_best_iteration':bi_l,'xgb_best_iteration':bi_x,'cat_best_iteration':bi_c,'ordinal_best_iterations':bi_o,'repair_rules':pre.report.to_dict('records') if pre.report is not None else []}
    json.dump(meta,open(mp,'w'),indent=2,default=str)
    print(pd.DataFrame(rec_rows).to_string(index=False)); print('selected',len(selected),'saved',pp)
    del raw_tr,raw_va,raw_te,btr,bva,bte,Xtr,Xva,Xte,A,B,C; gc.collect()
    if torch is not None and CUDA_AVAILABLE: torch.cuda.empty_cache()



FOLD 0 


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning gen

 fold                   feature  observed_outer_train  remaining_missing_train  remaining_missing_valid  remaining_missing_test  mask_probe_n  accuracy  majority_accuracy  log_loss backend devices
    0     general_health_status                 69734                     5996                     1560                    1859          1390  0.479856           0.353237  1.113020     GPU       0
    0      depression_frequency                 69497                     6233                     1532                    1931          1392  0.638649           0.506466  0.853456     GPU       0
    0 worry_about_medical_costs                 69683                     6047                     1534                    1800          1392  0.631466           0.566092  0.807969     GPU       0
    0        food_security_4cat                 69514                     6216                     1480                    1948          1396  0.997851           0.840974  0.012037     GPU       0
selected 181 sa

## 6. Honest OOF evaluation, subgroup diagnosis, and preregistered blends

Standalone V15 is diagnostic only. The only submission candidates are the prespecified Recovery blends: a global blend and a missingness-gated blend. For each held-out fold, alpha and QWK thresholds are fit using only the other four folds.


In [9]:
denoised_oof=np.full(len(y),np.nan,float); coverage=np.zeros(len(y),np.int8)
test_fold=[]; gate_oof=np.zeros(len(y),bool); gate_test_fold=[]
rec_diag=[]
for f in range(5):
    pp,mp,rp,fp=fold_paths(f); z=np.load(pp,allow_pickle=False)
    idx=pd.Index(train_ids).get_indexer(z['valid_id'])
    if (idx<0).any() or not np.all(folds[idx]==f): raise RuntimeError(f'fold {f} ID/fold contract failure')
    denoised_oof[idx]=z['valid_pred']; coverage[idx]+=1; gate_oof[idx]=z['valid_gate'].astype(bool)
    if not np.array_equal(z['test_id'],test_ids): raise RuntimeError(f'fold {f} test ID drift')
    test_fold.append(z['test_pred']); gate_test_fold.append(z['test_gate'].astype(bool))
    rec_diag.append(pd.read_csv(rp))
assert np.all(coverage==1) and np.isfinite(denoised_oof).all()
denoised_test=np.mean(np.vstack(test_fold),axis=0)
gate_test=(np.mean(np.vstack(gate_test_fold),axis=0)>=0.5)
reconstruction_df=pd.concat(rec_diag,ignore_index=True); reconstruction_df.to_csv(OUT/'reconstruction_metrics_v15.csv',index=False)

# Standalone cross-fitted calibration.
den_cls=np.zeros(len(y),int); den_thr={}
for f in range(5):
    tr=folds!=f; va=~tr; t=fit_thr(y[tr],denoised_oof[tr]); den_cls[va]=disc(denoised_oof[va],t); den_thr[f]=t

# Prespecified blend candidates with nested held-out-fold alpha/threshold choice.
def nested_blend(gated):
    score=np.full(len(y),np.nan,float); cls=np.zeros(len(y),int); rows=[]; thrs={}
    for f in range(5):
        tr=folds!=f; va=~tr
        best=None
        for a in CFG.alpha_grid:
            cand_tr=recovery_oof[tr].copy()
            if gated:
                gm=gate_oof[tr]; cand_tr[gm]=(1-a)*recovery_oof[tr][gm]+a*denoised_oof[tr][gm]
            else: cand_tr=(1-a)*recovery_oof[tr]+a*denoised_oof[tr]
            t=fit_thr(y[tr],cand_tr); s=qwk(y[tr],disc(cand_tr,t))
            if best is None or s>best['train_qwk']: best={'alpha':float(a),'train_qwk':float(s),'thr':t}
        a=best['alpha']; cand_va=recovery_oof[va].copy()
        if gated:
            gm=gate_oof[va]; cand_va[gm]=(1-a)*recovery_oof[va][gm]+a*denoised_oof[va][gm]
        else: cand_va=(1-a)*recovery_oof[va]+a*denoised_oof[va]
        score[va]=cand_va; cls[va]=disc(cand_va,best['thr']); thrs[f]=best['thr']
        rows.append({'fold':f,'alpha':a,'train_qwk_for_alpha':best['train_qwk'],'t1':best['thr'][0],'t2':best['thr'][1],'t3':best['thr'][2]})
    return score,cls,pd.DataFrame(rows),thrs

global_score,global_cls,global_sel,global_thr=nested_blend(False)
gated_score,gated_cls,gated_sel,gated_thr=nested_blend(True)
global_sel.to_csv(OUT/'global_blend_fold_selection.csv',index=False); gated_sel.to_csv(OUT/'gated_blend_fold_selection.csv',index=False)

# Reconstruction-affected subgroup is defined fold-locally after deterministic repair.
print('OOF reconstructed critical subgroup',gate_oof.mean(),'test majority-gate subgroup',gate_test.mean())


OOF reconstructed critical subgroup 0.28534908042212903 test majority-gate subgroup 0.2830643116707513


In [10]:
def evaluate(name,score,cls):
    fold_delta=[]
    for f in range(5):
        m=folds==f; fold_delta.append(qwk(y[m],cls[m])-qwk(y[m],recovery_class[m]))
    rng=np.random.default_rng(SEED+1515); boot=np.empty(CFG.bootstrap_reps)
    for b in range(CFG.bootstrap_reps):
        ix=rng.integers(0,len(y),len(y)); boot[b]=qwk(y[ix],cls[ix])-qwk(y[ix],recovery_class[ix])
    np.save(OUT/f'bootstrap_{name}_v15.npy',boot)
    crit=gate_oof; comp=~gate_oof
    def subgroup_auc(mask,p):
        m=mask & np.isin(y,[1,2])
        return float(roc_auc_score((y[m]>1).astype(int),p[m])) if len(np.unique(y[m]))>1 else np.nan
    e={
        'name':name,'qwk':qwk(y,cls),'recovery_qwk':qwk(y,recovery_class),'delta_qwk':qwk(y,cls)-qwk(y,recovery_class),
        'auc_1v2':auc12(y,score),'recovery_auc_1v2':auc12(y,recovery_oof),'delta_auc_1v2':auc12(y,score)-auc12(y,recovery_oof),
        'auc_ygt1':aucgt1(y,score),'delta_auc_ygt1':aucgt1(y,score)-aucgt1(y,recovery_oof),
        'critical_missing_auc_1v2':subgroup_auc(crit,score),'critical_missing_recovery_auc_1v2':subgroup_auc(crit,recovery_oof),
        'complete_auc_1v2':subgroup_auc(comp,score),'complete_recovery_auc_1v2':subgroup_auc(comp,recovery_oof),
        'rmse':float(np.sqrt(mean_squared_error(y,score))),'mae':float(mean_absolute_error(y,score)),'spearman':float(spearmanr(y,score)[0]),
        'prediction_corr':float(np.corrcoef(score,recovery_oof)[0,1]),'residual_corr':float(np.corrcoef(score-y,recovery_oof-y)[0,1]),
        'positive_folds':int(np.sum(np.asarray(fold_delta)>0)),'min_fold_delta':float(np.min(fold_delta)),'fold_deltas':list(map(float,fold_delta)),
        'bootstrap_p_gt0':float(np.mean(boot>0)),'bootstrap_median':float(np.median(boot)),'bootstrap_95':list(map(float,np.quantile(boot,[.025,.975])))
    }
    e['critical_missing_delta_auc_1v2']=e['critical_missing_auc_1v2']-e['critical_missing_recovery_auc_1v2']
    e['complete_delta_auc_1v2']=e['complete_auc_1v2']-e['complete_recovery_auc_1v2']
    return e

E_DEN=evaluate('denoised_standalone',denoised_oof,den_cls)
E_GLOBAL=evaluate('recovery_global_blend',global_score,global_cls)
E_GATED=evaluate('recovery_missingness_gated_blend',gated_score,gated_cls)
evals=[E_DEN,E_GLOBAL,E_GATED]
flat=pd.DataFrame([{k:v for k,v in e.items() if not isinstance(v,list)} for e in evals])
flat.to_csv(OUT/'candidate_metrics_v15.csv',index=False)
print(flat.to_string(index=False))

# Only the two Recovery sidecars are eligible for submission.
eligible=[E_GLOBAL,E_GATED]
best=max(eligible,key=lambda e:(e['delta_qwk'],e['critical_missing_delta_auc_1v2'],e['delta_auc_1v2']))
passes=bool(
    best['delta_qwk']>=CFG.min_delta_qwk and
    best['positive_folds']>=CFG.min_positive_folds and
    best['bootstrap_p_gt0']>=CFG.min_boot_p and
    best['delta_auc_1v2']>=CFG.min_delta_auc_1v2 and
    best['critical_missing_delta_auc_1v2']>=CFG.min_critical_missing_delta_auc_1v2 and
    best['complete_delta_auc_1v2']>=CFG.min_complete_delta_auc_1v2
)
print('\nBEST ELIGIBLE',best['name'],'SUBMISSION GATE',passes)
print(json.dumps(best,indent=2))


                            name      qwk  recovery_qwk  delta_qwk  auc_1v2  recovery_auc_1v2  delta_auc_1v2  auc_ygt1  delta_auc_ygt1  critical_missing_auc_1v2  critical_missing_recovery_auc_1v2  complete_auc_1v2  complete_recovery_auc_1v2     rmse      mae  spearman  prediction_corr  residual_corr  positive_folds  min_fold_delta  bootstrap_p_gt0  bootstrap_median  critical_missing_delta_auc_1v2  complete_delta_auc_1v2
             denoised_standalone 0.475999      0.481867  -0.005868 0.765245          0.765796      -0.000552  0.779408       -0.000572                  0.754939                           0.755215          0.769267                   0.769925 0.515819 0.426619  0.508981         0.993843       0.997682               0       -0.008308           0.0000         -0.005907                       -0.000276               -0.000658
           recovery_global_blend 0.481227      0.481867  -0.000640 0.765905          0.765796       0.000108  0.780082        0.000102                  

## 7. Test construction and artifact export

The final test alpha is the median of the five held-out-fold selected coarse alphas. Thresholds are the median of the five fold-local threshold vectors. For the gated candidate, a test respondent is gated only when at least three of five fold-local deterministic-repair views still require reconstruction of a critical question.


In [11]:
def final_candidate(name):
    if name=='recovery_global_blend':
        a=float(np.median(global_sel.alpha.to_numpy())); score=(1-a)*recovery_test+a*denoised_test
        t=np.median(np.vstack([global_thr[f] for f in range(5)]),axis=0); return score,disc(score,t),a,t
    if name=='recovery_missingness_gated_blend':
        a=float(np.median(gated_sel.alpha.to_numpy())); score=recovery_test.copy(); score[gate_test]=(1-a)*recovery_test[gate_test]+a*denoised_test[gate_test]
        t=np.median(np.vstack([gated_thr[f] for f in range(5)]),axis=0); return score,disc(score,t),a,t
    raise KeyError(name)

recommended=None; final_alpha=None; final_thresholds=None
best_score,best_class,final_alpha,final_thresholds=final_candidate(best['name'])

# Always export score diagnostics; a submission CSV is gated.
pd.DataFrame({
    'id':train_ids,'life_satisfaction':y,'fold':folds,'critical_reconstructed':gate_oof.astype(np.int8),
    'recovery_score':recovery_oof,'recovery_class':recovery_class,
    'denoised_score':denoised_oof,'denoised_class':den_cls,
    'global_blend_score':global_score,'global_blend_class':global_cls,
    'gated_blend_score':gated_score,'gated_blend_class':gated_cls
}).to_csv(OUT/'oof_v15.csv',index=False)

pd.DataFrame({
    'id':test_ids,'critical_reconstructed_majority':gate_test.astype(np.int8),
    'recovery_score':recovery_test,'denoised_score':denoised_test,'best_candidate_score':best_score,'best_candidate_class':best_class
}).to_csv(OUT/'test_scores_v15.csv',index=False)

if passes:
    sub=pd.DataFrame({'id':test_ids,'life_satisfaction':best_class.astype(int)})
    assert len(sub)==CFG.expected_test_ids and sub.id.is_unique and sub.life_satisfaction.isin([1,2,3,4]).all() and sub.isna().sum().sum()==0
    recommended=OUT/'submission_v15_recommended.csv'; sub.to_csv(recommended,index=False)

summary={
    'version':'v15_fold_safe_probabilistic_survey_denoising','timestamp_utc':pd.Timestamp.utcnow().isoformat(),
    'contract_hash':CFG_HASH,'config':asdict(CFG),'data_dir':str(DATA),'historical_artifacts':STORE.source(),
    'recovery_public_lb':CFG.recovery_public_lb,'recovery_oof_qwk':RECOVERY_QWK,
    'critical_features':CRITICAL,'reconstruction_metrics':reconstruction_df.to_dict('records'),
    'denoised_standalone':E_DEN,'global_blend':E_GLOBAL,'gated_blend':E_GATED,
    'best_eligible_candidate':best['name'],'submission_gate':passes,
    'final_alpha':final_alpha,'final_thresholds':list(map(float,final_thresholds)),
    'recommended_submission':None if recommended is None else str(recommended),
    'recovery_control_submission':str(OUT/'submission_v15_recovery_control.csv'),
    'constraints':[
        'Competition task data only; no external respondent rows or labels.',
        'Unique respondent entity; exact duplicate IDs checked and collapsed.',
        'Frozen V2 primary five-fold assignment and exact historical Recovery control.',
        'Deterministic repair rules learned on outer-train only.',
        'Reconstructors never see life_satisfaction and exclude id/household_id and the feature being reconstructed.',
        'Posterior features are exact one-hot for observed cells and model probabilities only for cells still missing after deterministic repair.',
        'Special survey codes remain categorical probability mass; expected-value posterior uses valid ordered codes only.',
        'Feature selection and model early stopping use only an inner subset of outer-train; outer-validation target is untouched until scoring.',
        'V15 downstream uses fixed V2-style core mean plus 0.45 cumulative-ordinal sidecar; no weight search inside the denoised model.',
        'Only coarse Recovery blend alphas 0.10/0.20/0.30/0.40 are considered; fold-held-out scoring is used for reporting.',
        'Recommended submission is emitted only if preregistered QWK, fold stability, bootstrap, boundary, and missing-subgroup gates pass.'
    ]
}
with open(OUT/'v15_summary.json','w') as f: json.dump(summary,f,indent=2,default=lambda o:o.item() if hasattr(o,'item') else str(o))

STORE.close()
print('\nFINAL STATUS:', 'SUBMISSION READY' if passes else 'DO NOT SUBMIT — V15 gate failed')
print('Recovery control:',OUT/'submission_v15_recovery_control.csv')
print('Recommended:',recommended)
print('Summary:',OUT/'v15_summary.json')
print('\nArtifacts:')
for p in sorted(OUT.iterdir()): print(' -',p.name)



FINAL STATUS: DO NOT SUBMIT — V15 gate failed
Recovery control: /kaggle/working/oqdq5_v15/submission_v15_recovery_control.csv
Recommended: None
Summary: /kaggle/working/oqdq5_v15/v15_summary.json

Artifacts:
 - bootstrap_denoised_standalone_v15.npy
 - bootstrap_recovery_global_blend_v15.npy
 - bootstrap_recovery_missingness_gated_blend_v15.npy
 - candidate_metrics_v15.csv
 - fold0_features.csv
 - fold0_meta.json
 - fold0_predictions.npz
 - fold0_reconstruction.csv
 - fold1_features.csv
 - fold1_meta.json
 - fold1_predictions.npz
 - fold1_reconstruction.csv
 - fold2_features.csv
 - fold2_meta.json
 - fold2_predictions.npz
 - fold2_reconstruction.csv
 - fold3_features.csv
 - fold3_meta.json
 - fold3_predictions.npz
 - fold3_reconstruction.csv
 - fold4_features.csv
 - fold4_meta.json
 - fold4_predictions.npz
 - fold4_reconstruction.csv
 - gated_blend_fold_selection.csv
 - global_blend_fold_selection.csv
 - oof_v15.csv
 - reconstruction_metrics_v15.csv
 - submission_v15_recovery_control.c